In [1]:
import torch
import pandas as pd
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
import warnings
warnings.filterwarnings('ignore')

print(f"GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Modelo de GPU: {torch.cuda.get_device_name(0)}")

corpus = pd.read_csv('C:/Users/inqui/OneDrive/Desktop/Clases/26-2/LLM_PROJECT_1/datos/processed/tiktok_infraes_limpio.csv',
                     usecols = ['text'],encoding = 'utf-8-sig')

GPU disponible: False


In [11]:
analizador = pipeline(
    "sentiment-analysis",
#    model = 'citizenlab/distilbert-base-multilingual-cased-toxicity' #podria funcionar con un tuneo
#    model ="BAAI/bge-reranker-v2-m3" #no sirve para el objetivo
#    model="distilbert-base-uncased-finetuned-sst-2-english" #el que usa el profe. podria funcionar tuneado
    model = 'nlptown/bert-base-multilingual-uncased-sentiment', #podria funcionar con un tuneo, el mas prometedor por ahora    
#    model = "FacebookAI/roberta-large-mnli", #puede prometer
#    model = 'FacebookAI/xlm-roberta-large', #demasiado crudo
#    model = 'cardiffnlp/twitter-roberta-base-sentiment-latest', #podria ser, pero hay que enseñarle español
#    model = 'Bhumika/roberta-base-finetuned-sst2',
    torch_dtype="auto",           # Detecta automáticamente el tipo óptimo
#    device_map="auto"             # Coloca el modelo en GPU si está disponible
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [13]:
etiqueta = []
confianza = []

for texto in list(corpus['text']):
    aux = analizador(texto)[0]
    etiqueta.append(aux['label'])
    confianza.append(aux['score'])

In [14]:
resultados = pd.DataFrame([list(corpus['text']),etiqueta,confianza])
resultados[0:10]

,0,1,2,3,4,5,6,7,8,9,...,814,815,816,817,818,819,820,821,822,823
0,Y si mejor arreglan los hospitales? Ponen mas ...,Mejor que pongan botes de basura por todo refo...,Sin mantenimiento estarán en cuanto acabe el m...,Dinero desperdiciado,y los medicamentos en insumos en los hospitale...,"esos baños deberian estar en el metro, no sabe...",para que después huela horrible,Baños? yo veo hoteles 🗿,"Lástima que una vez que se acabe el mundial, n...",Nmmn arreglen el sistema de drenaje q se tapa ...,...,"para los que se quejan de los baños así, segur...",Van a terminar vandalizados y por lo regular s...,No deverian estar cerca del estadio,Weeeeey está bien exactamente para eso para ne...,ni los baños públicos tendrán activación econó...,Me recuerda a las mamás (no todas) cuando va a...,Cuando meterán presupuesto para medicamentos e...,"la gente dice, no me gustan, prefiero cagar en...",no al mundial por el ebola,"Mejor que inviertan en medicamentos, búsqueda ..."
1,2 stars,5 stars,1 star,1 star,2 stars,1 star,1 star,1 star,1 star,4 stars,...,3 stars,1 star,1 star,3 stars,1 star,3 stars,3 stars,2 stars,1 star,5 stars
2,0.343795,0.580883,0.560409,0.777052,0.304614,0.329172,0.58878,0.324706,0.559407,0.404197,...,0.397334,0.674747,0.443987,0.452643,0.534318,0.449632,0.244966,0.464355,0.595486,0.5506


In [12]:
ejemplos = [
    "This movie is absolutely amazing! I loved every minute.",
    "Terrible experience. Worst film I've ever seen.",
    "It was okay, nothing special but not terrible either.",
    "The acting was great but the plot was confusing."
]

print("Clasificando sentimientos:\n")
for texto in ejemplos:
    resultado = analizador(texto)[0]
    label = resultado['label']
    score = resultado['score']
    emoji = "😊" if label == "POSITIVE" else "😞"
    print(resultado)
    print(f"{emoji} [{label}] (confianza: {score:.2f})")
    print(f"   Texto: {texto[:70]}...")
    print()

Clasificando sentimientos:

{'label': '5 stars', 'score': 0.9732953906059265}
😞 [5 stars] (confianza: 0.97)
   Texto: This movie is absolutely amazing! I loved every minute....

{'label': '1 star', 'score': 0.9537760019302368}
😞 [1 star] (confianza: 0.95)
   Texto: Terrible experience. Worst film I've ever seen....

{'label': '3 stars', 'score': 0.8621777296066284}
😞 [3 stars] (confianza: 0.86)
   Texto: It was okay, nothing special but not terrible either....

{'label': '3 stars', 'score': 0.553006112575531}
😞 [3 stars] (confianza: 0.55)
   Texto: The acting was great but the plot was confusing....

